<a href="https://colab.research.google.com/github/Heba2627/NLP-Project/blob/main/NLP_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importing Dataset via. Pandas

In [9]:
import pandas as pd
import numpy as np
import regex as re

# Login using e.g. `huggingface-cli login` to access this dataset
splits = {'train': 'train.parquet', 'validation': 'validation.parquet', 'test': 'test.parquet'}

In [10]:
#Fetching train, validation and test dataframes.
train_df = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["train"])
val_df   = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["validation"])
test_df  = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["test"])

#I NEED TO REPLACE ALL THE "df" REFERENCES WITH "train_df" AT SOME POINT.
df = train_df
print(df.columns)
answerable_df = df[df["answerable"] == True]
unanswerable_df = df[df["answerable"] == False]

print(len(answerable_df))
print(len(unanswerable_df))
print("% of dataset that is answerable:", len(answerable_df)/len(df)*100, "%")
print("% of dataset that is unanswerable:", len(unanswerable_df)/len(df)*100, "%")

Index(['question', 'context', 'lang', 'answerable', 'answer_start', 'answer',
       'answer_inlang'],
      dtype='object')
13962
1381
% of dataset that is answerable: 90.99915270807534 %
% of dataset that is unanswerable: 9.000847291924655 %


In [11]:
#Check for letters in all languages + Mark for arabic
word_pattern = re.compile(r"[\p{L}\p{M}]+", flags=re.UNICODE)

def n_tokens(text):
    return len(word_pattern.findall(str(text)))
def median_iqr(s):
    q1, median, q3 = np.percentile(s, [25, 50, 75])
    return pd.Series({"median": median, "Q1": q1, "Q3": q3, "IQR": q3 - q1})

df["question_len"] = df["question"].apply(n_tokens)
df["context_len"] = df["context"].apply(n_tokens)

#Approach idea: build a mini DataFrame to obtain a table like described in the project pdf.
length_stats = pd.DataFrame({"question_length": median_iqr(df["question_len"]), "context_length": median_iqr(df["context_len"])})
length_stats

,question_length,context_length
median,5.0,84.0
Q1,4.0,56.0
Q3,7.0,122.0
IQR,3.0,66.0


In [12]:
#missing values
missing = df.isna().sum().to_frame("n_missing")
missing["pct_missing"] = (100 * missing["n_missing"]/len(df))
missing

#duplicates
dup_filter = df.duplicated(subset=["question", "context"], keep=False)
dup_amount = dup_filter.sum()
dup_groups = df[dup_filter].groupby(["question","context"]).ngroups
redudant_rows = df.duplicated(subset=["question", "context"], keep="first").sum()
print(dup_groups)
print(dup_amount)
print(redudant_rows)

17
34
17


# Tokenisation

In [54]:
!pip install deep_translator

In [55]:
from transformers import AutoTokenizer
from deep_translator import GoogleTranslator
from collections import Counter, defaultdict

tok = AutoTokenizer.from_pretrained("xlm-roberta-base")

#We need to root out symbols so check for letter
has_letter = re.compile(r"\p{L}", flags=re.UNICODE)

#NOTE: We will probably need a special case for japanese since its the only
#language that doesnt have whitespace or punctuation!
def tokenize_words_xlmr(text):
    pieces = tok.tokenize(str(text))
    words, current = [], ""
    for p in pieces:
        if p.startswith("▁"):
            if current != "":
                words.append(current)
            current = p[1:]
        else:
            current += p
    if current != "":
        words.append(current)
    return [w.lower() for w in words if w != "" and has_letter.search(w)]

def translate(token, lang):
    try:
      return GoogleTranslator(source=lang, target="en").translate(token)
    except:
      return None

counters = defaultdict(Counter)
for lang, question in zip(df["lang"], df["question"]):
    counters[lang].update(tokenize_words_xlmr(question))

rows = [
    {"lang": lang, "token": token, "count": count}
    for lang, counter in counters.items()
    for token, count in counter.most_common(5)
]
top5_df = pd.DataFrame(rows)
top5_df = top5_df[top5_df["lang"].isin(["ar", "ko", "te"])]
top5_df["In English"] = [translate(token, lang) for token, lang in zip(top5_df["token"], top5_df["lang"])]
top5_df = top5_df[["lang", "token", "In English", "count"]]
top5_df

,lang,token,In English,count
10,ko,가장,None,527
11,ko,무엇인가?,None,495
12,ko,언제,None,336
13,ko,몇,None,234
14,ko,어디인가?,None,228
25,ar,في,None,592
26,ar,من,None,584
27,ar,متى,None,535
28,ar,ما,None,441
29,ar,هو,None,349


In [15]:
#Task 3
def verify_answerable(df):
    checked = 0
    failures = []
    for i, row in enumerate(df):
        if row["answerable"]:
            checked+= 1
            context=row["context"]
            answer=row["answer"]
            start=row["answer_start"]

            expected=context[start:start+len(answer)]
            if expected!=answer:
                failures.append({
                    "index": i,
                    "answer_start": start,
                    "answer": answer,
                    "substring_found": expected,
                    "context": context})
    return checked,failures

#The pdf said to apply it to both splits
#we have to use ar,ko and te only, and they exist only in train and validation.
checked_train,failures_train=verify_answerable(train_df.to_dict("records"))
print(f"Checked {checked_train} answerable rows in the training set, found {len(failures_train)} failures")

checked_val,failures_val=verify_answerable(val_df.to_dict("records"))
print(f"Checked {checked_val} answerable rows in the validation set, found {len(failures_val)} failures")


Checked 13962 answerable rows in the training set, found 0 failures
Checked 2313 answerable rows in the validation set, found 0 failures


# i) Majority Baseline

In [16]:
majority_label = df["answerable"].value_counts().idxmax()
print("Majority class form training split: ", majority_label)

def majority_baseline(question=None, Context=None):
    return majority_label

Majority class form training split:  True


In [17]:
val_pred = [majority_baseline(q, c) for q, c in zip(val_df["question"], val_df["context"])]
acc_score = (pd.Series(val_pred).values == val_df["answerable"].values).mean()
print("Accuracy of Majority Baseline on Validation split: ", acc_score*100, "%")

Accuracy of Majority Baseline on Validation split:  76.81833277980738 %


## ii) Determinstic rule-based classifier baseline

- Raw tokenization overlapping between questions and contexts would never work(different languages)
- Whatever approach I take will struggle with the semintically equivalent words like created and invented, as we can't have semantic understanding without a model(unless I bring a synonym dictionary)
- What I am going to do is to basically translate the question using deep-translator's google translator, tokenize the question and the context, removing stop words, put into lower case, and possibly lemmatize, and then I will depend on lexical overlap between the tokens but not the pure thing(because of the semantically equivalent words problem), so I will try to map some question words to their expected answers using regex, and then I will use the overlap score between the question and the context to determine the answerability.
- What it can exploit is the samples that survives the translation, named entities, numbers, dates, and question-word typing. These will survive because proper nouns and numerals are often unchanged, and question-word categories are a small closed set you can pattern-match directly.
- What it can't exploit is the mistranslations, negations and basically anyhting requires semantic understanding.

In [18]:
import string

tokenizer= AutoTokenizer.from_pretrained("xlm-roberta-base")

STOPWORDS= {
    "the", "a", "an", "is", "are", "was", "were", "of", "in", "on", "at",
    "to", "for", "and", "or", "that", "this", "these", "those", "by", "with",
    "from", "as", "it", "its"
}

QUESTION_TYPE_PATTERNS= {
    "person":re.compile(r"\bwho\b", re.I),
    "date":re.compile(r"\bwhen\b", re.I),
    "location":re.compile(r"\bwhere\b", re.I),
    "number":re.compile(r"\bhow many\b|\bhow much\b", re.I),
}

ENTITY_PATTERNS= {
    "person":re.compile(r"\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)+\b"),
    "date":re.compile(r"\b\d{1,2}(st|nd|rd|th)?\s\w+\s\d{4}\b|\b\d{4}\b|\b\w+\s\d{1,2},\s\d{4}\b"),
    "location":re.compile(r"\b[A-Z][a-z]+\b"),
    "number":re.compile(r"\b\d+([.,]\d+)?\b"),
}


def translate_to_english(text,lang):
    try:
        return GoogleTranslator(source=lang, target="en").translate(text)
    except Exception:
        return text


def clean_tokens(text):
    tokens= tokenizer.tokenize(text.lower())
    tokens= [token for token in tokens if token and token not in string.punctuation]
    tokens= [token for token in tokens if token not in STOPWORDS]
    return set(tokens)


def lexical_overlap_score(question_en, context):
    q_tokens= clean_tokens(question_en)
    c_tokens= clean_tokens(context)
    if not q_tokens:
        return 0.0
    return len(q_tokens&c_tokens)/len(q_tokens)


def detect_question_type(question_en):
    for qtype, pattern in QUESTION_TYPE_PATTERNS.items():
        if pattern.search(question_en):
            return qtype
    return "other"


def context_has_type_match(context, qtype):
    if qtype == "other":
        return False
    pattern= ENTITY_PATTERNS.get(qtype)
    return bool(pattern.search(context)) if pattern else False


def predict_answerable(question, context, source_lang="auto"):
    question_en= translate_to_english(question, lang=source_lang)
    overlap= lexical_overlap_score(question_en, context)
    qtype= detect_question_type(question_en)
    type_match= context_has_type_match(context, qtype)
    prediction= (overlap>=0.1) or type_match
    return prediction

In [19]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
validation_df= val_df[val_df["lang"].isin(["ar", "ko", "te"])]

def evaluate_classifier(val_df):
    y_true, y_pred= [], []

    for row in val_df.to_dict("records"):
        pred = predict_answerable(
            row["question"], row["context"],
            source_lang=row["lang"],
        )
        y_true.append(bool(row["answerable"]))
        y_pred.append(pred)



    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }
    return metrics



metrics= evaluate_classifier(validation_df)
print(f"Accuracy:{metrics['accuracy']:.3f}")
print(f"Precision:{metrics['precision']:.3f}")
print(f"Recall:{metrics['recall']:.3f}")
print(f"F1:{metrics['f1']:.3f}")
print(f"Confusion matrix [[TN, FP], [FN, TP]]: {metrics['confusion_matrix']}")

#NOTE: Check the max seq length problem

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (611 > 512). Running this sequence through the model will result in indexing errors


Accuracy:0.318
Precision:0.936
Recall:0.220
F1:0.356
Confusion matrix [[TN, FP], [FN, TP]]: [[149, 15], [773, 218]]


# Week 37:

## BIO Labeling of the training split

In [50]:
#CODE CELL WAS IMPORTED FROM lab2.ipynb
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", re.UNICODE)


def tokenize_with_offsets(text):
    matches = list(TOKEN_PATTERN.finditer(text))
    tokens = [match.group(0) for match in matches]
    offsets = [(match.start(), match.end()) for match in matches]
    return tokens, offsets


def character_span_to_bio(context, answer_start=None, answer_text=""):
    tokens, offsets = tokenize_with_offsets(context)
    labels = ["O"] * len(tokens)

    if answer_start is None or answer_text == "":
        return tokens, offsets, labels

    answer_end = answer_start + len(answer_text)
    if context[answer_start:answer_end] != answer_text:
        raise ValueError("The supplied answer text does not match the character span")

    covered = [
        index
        for index, (start, end) in enumerate(offsets)
        if start < answer_end and end > answer_start
    ]
    if not covered:
        raise ValueError("The answer does not overlap any token")
    if offsets[covered[0]][0] != answer_start or offsets[covered[-1]][1] != answer_end:
        raise ValueError("The answer span does not align with token boundaries")

    labels[covered[0]] = "B-ANS"
    for index in covered[1:]:
        labels[index] = "I-ANS"
    return tokens, offsets, labels


def bio_to_character_span(context, offsets, labels):
    if len(offsets) != len(labels):
        raise ValueError("Offsets and labels must have the same length")
    if not set(labels) <= {"O", "B-ANS", "I-ANS"}:
        raise ValueError("Only O, B-ANS and I-ANS labels are supported")

    answer_indices = [
        index for index, label in enumerate(labels) if label != "O"
    ]
    if not answer_indices:
        return None, ""

    start_index = answer_indices[0]
    expected_indices = list(range(start_index, start_index + len(answer_indices)))
    expected_labels = ["B-ANS"] + ["I-ANS"] * (len(answer_indices) - 1)
    if answer_indices != expected_indices:
        raise ValueError("The labels contain multiple or non-contiguous answer spans")
    if [labels[index] for index in answer_indices] != expected_labels:
        raise ValueError("The answer span must begin with B-ANS and continue with I-ANS")

    start = offsets[answer_indices[0]][0]
    end = offsets[answer_indices[-1]][1]
    return start, context[start:end]

In [64]:
df = df.copy()

df = df.drop(columns=["tokens", "offsets", "labels", "alignable"], errors="ignore")

def try_bio(row):
    if not row["answerable"]:
        # Unanswerable: always succeeds, labels are all "O"
        tokens, offsets, labels = character_span_to_bio(row["context"], None, "")
        return pd.Series({"tokens": tokens, "offsets": offsets, "labels": labels, "alignable": True})
    try:
        tokens, offsets, labels = character_span_to_bio(
            row["context"], row["answer_start"], row["answer"]
        )
        return pd.Series({"tokens": tokens, "offsets": offsets, "labels": labels, "alignable": True})
    except ValueError:
        # Answer text doesn't align to token boundaries we flag it.
        return pd.Series({"tokens": None, "offsets": None, "labels": None, "alignable": False})

#Appending BIO columns to df
bio_results = df.apply(try_bio, axis=1)
df = pd.concat([df, bio_results], axis=1)

print(f"{(~df['alignable']).sum()} / {len(df)} rows fail alignment")
print(df.groupby("answerable")["alignable"].mean())


# Derive the clean training view by filtering out the failing rows
BIO_df = df[df["alignable"]].reset_index(drop=True)
dropped_df = df[~df["alignable"]]  # kept around for reporting


# Round-trip check: after BIO labeling, is the char length the same for tokens and labels?
mismatches = 0
for row in BIO_df.itertuples():
    if row.answerable:
        rt_start, rt_text = bio_to_character_span(row.context, row.offsets, row.labels)
        if rt_start != row.answer_start or rt_text != row.answer:
            mismatches += 1
    else:
        rt_start, rt_text = bio_to_character_span(row.context, row.offsets, row.labels)
        if rt_start is not None or rt_text != "":
            mismatches += 1

print(f"{mismatches} / {len(BIO_df)} rows failed round-trip check")

BIO_df[["context", "answerable", "tokens", "labels"]]

178 / 15343 rows fail alignment
answerable
False    1.000000
True     0.987251
Name: alignable, dtype: float64
0 / 15165 rows failed round-trip check


,context,answerable,tokens,labels
0,WikiLeaks () is an international non-profit or...,True,"[WikiLeaks, (, ), is, an, international, non, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
1,The war in Europe concluded with an invasion o...,True,"[The, war, in, Europe, concluded, with, an, in...","[O, O, O, O, O, O, O, O, O, B-ANS, O, O, O, O,..."
2,Same-sex marriage in the United States expande...,False,"[Same, -, sex, marriage, in, the, United, Stat...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
3,The exact number of Arab casualties is unknown...,True,"[The, exact, number, of, Arab, casualties, is,...","[O, O, O, O, O, O, O, B-ANS, O, O, O, O, O, O,..."
4,"As Thomas Hall (2000) notes, ""The Sung Empire ...",True,"[As, Thomas, Hall, (, 2000, ), notes, ,, "", Th...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
...,...,...,...,...
15160,"In February 2012, Somali government officials ...",True,"[In, February, 2012, ,, Somali, government, of...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
15161,The first earth tracks were created by humans ...,True,"[The, first, earth, tracks, were, created, by,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
15162,"Abdel Fattah Saeed Hussein Khalil El-Sisi ( """"...",True,"[Abdel, Fattah, Saeed, Hussein, Khalil, El, -,...","[B-ANS, I-ANS, I-ANS, I-ANS, I-ANS, I-ANS, I-A..."
15163,Munich (; ; ) is the capital and most populous...,True,"[Munich, (, ;, ;, ), is, the, capital, and, mo...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."


In [69]:
#A view of the rows where answerable = true but the length of the answer does not match
#with answer_start + len(answer)
dropped_df[["context", "answerable", "answer"]]

,context,answerable,answer
28,With the success of the Indian Navy's operatio...,True,India
68,"The Wright brothers, Orville (August 19, 1871 ...",True,America
113,The genocide in Bangladesh began on 26 March 1...,True,e
154,Putin studied Law at the Leningrad State Unive...,True,natoly Sobchak
282,"According to tradition, ""Vyasa"" is the compile...",True,Vyas
...,...,...,...
14777,"The tower is tall, about the same height as an...",True,ame height as an 81- building
14894,"According to the ""Oxford English Dictionary"", ...",True,"000,000"
14987,Her versatility is evident in Panchavataram on...,True,26 December 201
15000,Leonardo Del Vecchio started the company in 19...,True,Leonardo Del Vecchio started the company in 196


In [60]:
def run_bio_checks():
    checks_passed = 0
    checks_total = 0

    def check(name, condition):
        nonlocal checks_passed, checks_total
        checks_total += 1
        status = "PASS" if condition else "FAIL"
        if condition:
            checks_passed += 1
        print(f"[{status}] {name}")

    #Answer starting at character 0
    context = "Copenhagen is the capital of Denmark."
    answer_text = "Copenhagen"
    answer_start = 0
    tokens, offsets, labels = character_span_to_bio(context, answer_start, answer_text)
    check("answer at char 0: first token is B-ANS", labels[0] == "B-ANS")
    check("answer at char 0: no other tokens are ANS", labels[1:] == ["O"] * (len(labels) - 1))
    rt_start, rt_text = bio_to_character_span(context, offsets, labels)
    check("answer at char 0: round-trips correctly", (rt_start, rt_text) == (answer_start, answer_text))

    #Multi-token answer
    context = "The main character of Jujutsu Kaisen is Itadori Yuji."
    answer_text = "Itadori Yuji"
    answer_start = context.index(answer_text)
    tokens, offsets, labels = character_span_to_bio(context, answer_start, answer_text)
    ans_indices = [i for i, l in enumerate(labels) if l != "O"]
    check("multi-token answer: spans exactly 2 tokens", len(ans_indices) == 2)
    check("multi-token answer: starts with B-ANS", labels[ans_indices[0]] == "B-ANS")
    check("multi-token answer: continues with I-ANS", all(labels[i] == "I-ANS" for i in ans_indices[1:]))
    rt_start, rt_text = bio_to_character_span(context, offsets, labels)
    check("multi-token answer: round-trips correctly", (rt_start, rt_text) == (answer_start, answer_text))

    #Punctuation immediately adjacent to the answer (no whitespace separator)
    context = "She was born in Odense, Denmark."
    answer_text = "Odense"
    answer_start = context.index(answer_text)  # immediately followed by ','
    tokens, offsets, labels = character_span_to_bio(context, answer_start, answer_text)
    ans_indices = [i for i, l in enumerate(labels) if l != "O"]
    check("punctuation-adjacent answer: exactly one ANS token (comma excluded)", len(ans_indices) == 1)
    check("punctuation-adjacent answer: comma token is 'O'", tokens[ans_indices[-1] + 1] == "," and labels[ans_indices[-1] + 1] == "O")
    rt_start, rt_text = bio_to_character_span(context, offsets, labels)
    check("punctuation-adjacent answer: round-trips correctly", (rt_start, rt_text) == (answer_start, answer_text))

    #Unanswerable example (no answer_start / answer_text)
    context = "This string contains no answer for the question."
    tokens, offsets, labels = character_span_to_bio(context, answer_start=None, answer_text="")
    check("unanswerable: all labels are 'O'", all(l == "O" for l in labels))
    check("unanswerable: tokens/offsets still populated", len(tokens) > 0 and len(offsets) == len(tokens))
    rt_start, rt_text = bio_to_character_span(context, offsets, labels)
    check("unanswerable: round-trips to (None, '')", (rt_start, rt_text) == (None, ""))

    print(f"\n{checks_passed}/{checks_total} checks passed")
    assert checks_passed == checks_total, "One or more BIO conversion checks failed"

run_bio_checks()

[PASS] answer at char 0: first token is B-ANS
[PASS] answer at char 0: no other tokens are ANS
[PASS] answer at char 0: round-trips correctly
[PASS] multi-token answer: spans exactly 2 tokens
[PASS] multi-token answer: starts with B-ANS
[PASS] multi-token answer: continues with I-ANS
[PASS] multi-token answer: round-trips correctly
[PASS] punctuation-adjacent answer: exactly one ANS token (comma excluded)
[PASS] punctuation-adjacent answer: comma token is 'O'
[PASS] punctuation-adjacent answer: round-trips correctly
[PASS] unanswerable: all labels are 'O'
[PASS] unanswerable: tokens/offsets still populated
[PASS] unanswerable: round-trips to (None, '')

13/13 checks passed
